# 풀스택 GPT: #6.0 ~ #6.10
## Tasks:

- [x] Stuff Documents 체인을 사용하여 완전한 RAG 파이프라인을 구현하세요.
- [x] 체인을 수동으로 구현해야 합니다.
- [x] 체인에 ConversationBufferMemory를 부여합니다.
- [x] 이 문서를 사용하여 RAG를 수행하세요: [https://gist.github.com/serranoarevalo/5acf755c2b8d83f1707ef266b82ea223](https://gist.github.com/serranoarevalo/5acf755c2b8d83f1707ef266b82ea223)
- [x] 체인에 다음 질문을 합니다:
    > - Is Aaronson guilty?
    > - What message did he write in the table?
    > - Who is Julia?
- 다음과 같은 절차대로 구현하면 챌린지를 해결할 수 있습니다.
    > - [x] (1) 문서 로드하기 : TextLoader 등 을 사용해서 파일에서 텍스트를 읽어옵니다. [Document Loaders 관련 문서](https://python.langchain.com/v0.1/docs/modules/data_connection/document_loaders/)
    > - [x] (2) 문서 쪼개기 : CharacterTextSplitter 등 을 사용해서 문서를 작은 문서 조각들로 나눕니다. [Character Split 관련 문서](https://python.langchain.com/v0.1/docs/modules/data_connection/text_embedding/caching_embeddings/)
    > - [x] (3) 임베딩 생성 및 캐시 : OpenAIEmbeddings, CacheBackedEmbeddings 등 을 사용해 문서 조각들을 임베딩하고 임베딩을 저장합니다. Caching 관련 문서
    > - [x] (4) 벡터 스토어 생성 : FAISS 등 을 사용해서 임베딩된 문서들을 저장하고 검색할 수 있는 데이터베이스를 만듭니다. [FAISS 관련 문서](https://python.langchain.com/v0.1/docs/integrations/vectorstores/faiss/)
    > - [x] (5) 대화 메모리와 질문 처리 : ConversationBufferMemory를 사용해 대화 기록을 관리합니다.
    > - [x] (6) 체인 연결 : 앞에서 구현한 컴포넌트들을 적절하게 체인으로 연결합니다

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda
from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain.vectorstores import FAISS
from langchain.memory import ConversationBufferMemory

llm = ChatOpenAI(
    model="gpt-4o-mini-2024-07-18",
    temperature=0.2,)
loader = TextLoader("../files/document.txt")
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)
docs = loader.load_and_split(text_splitter=splitter)
embeddings = OpenAIEmbeddings()
cache_dir = LocalFileStore("../.cache/")
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)
vectorstore = FAISS.from_documents(docs, cached_embeddings)
memory = ConversationBufferMemory(return_messages=True)

retriever = vectorstore.as_retriever()
prompt = ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant. Answer questions using only the following context. if you don't know, don't make it up:\n\n{context}"),
    MessagesPlaceholder(variable_name="history"),
    ("human","{question}"),
])
def load_memory(input):
    return memory.load_memory_variables({})["history"]

def retrieve_docs(input_dict):
    return {"context": retriever.get_relevant_documents(input_dict["question"]), "question": input_dict["question"], "history": input_dict["history"]}
    
def invoke_chain(input):
    result = chain.invoke({"question": input})
    memory.save_context({"input": input}, {"output": result.content})

chain = RunnablePassthrough.assign(history=load_memory) | RunnableLambda(retrieve_docs) | prompt | llm

In [ ]:
invoke_chain("What message did he write in the table?")


memory is saved. input: What message did he write in the table?, output: Winston wrote the following messages on the table: 

1. "FREEDOM IS SLAVERY"
2. "TWO AND TWO MAKE FIVE"
3. "GOD IS POWER" 

These phrases reflect the Party's slogans and the distorted reality that Winston is being forced to accept.


In [10]:
invoke_chain("Who is Julia?")

memory is saved. input: Who is Julia?, output: Julia is a character in the context provided who has a romantic relationship with Winston. She is portrayed as someone who initially shares Winston's rebellious feelings against the Party, but later, under torture, Winston expresses that he has not betrayed her, indicating a deep emotional connection. Julia represents a personal and intimate connection for Winston amidst the oppressive regime of the Party.
